In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
from transformers import AutoModel,AutoConfig,AutoTokenizer,AutoModelForMultipleChoice

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
MODEL_NAME = 'microsoft/deberta-v2-xlarge'
MODEL_NAME_CHOICE = 'vinai/phobert-base'

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
from  transformers.modeling_outputs import MultipleChoiceModelOutput

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
from torch import nn

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
class CustomModelMultichoice(nn.Module):
    def __init__(self,config,num_choice):
        super(CustomModelMultichoice,self).__init__()
        model = AutoModelForMultipleChoice.from_config(config)
        model.classifier = nn.Linear(768,2)
        self.model = model
        ## add activation
        self.sigmoid = nn.Sigmoid()
        self.num_choice = num_choice
    def forward(self,input_ids = None,token_type_ids = None ,attention_mask = None,labels = None):
        outputs = self.model(input_ids=input_ids,token_type_ids=token_type_ids,attention_mask=attention_mask)
        logits = self.sigmoid(outputs.logits)
        loss = None
        if labels is not None:
            loss_func = nn.NLLLoss()
            loss = loss_func(logits.view(-1,self.num_choice),labels.view(-1))
        return MultipleChoiceModelOutput(loss = loss,logits=logits,hidden_states= None,attentions =None)

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
config = AutoConfig.from_pretrained(MODEL_NAME_CHOICE)

CustomModel = CustomModelMultichoice(config,3)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
prompt = "Bác Hồ là người nước nào ?."
candidate1 = "Việt Nam"
candidate2 = "Mỹ"
candidate3 = 'Việt Nam'

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_CHOICE)
inputs = tokenizer([[prompt, candidate1], [prompt, candidate2],[prompt, candidate3]], return_tensors="pt", padding=True)

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
import torch
labels = torch.tensor(0).unsqueeze(0)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
inputs['input_ids'] = inputs['input_ids'].unsqueeze(0)
inputs['token_type_ids'] = inputs['token_type_ids'].unsqueeze(0)
inputs['attention_mask'] = inputs['attention_mask'].unsqueeze(0)

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
CustomModel.eval()

CustomModelMultichoice(
  (model): RobertaForMultipleChoice(
    (roberta): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(64001, 768, padding_idx=1)
        (position_embeddings): Embedding(258, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-11): 12 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense):

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
target = torch.tensor([[1, 0, 1]])
target

tensor([[1, 0, 1]])

In [13]:
# --- [CELL 12]: ---
# cell_state: edited
# execution_status: {'status': 'error', 'done': True, 'execution_count': 13}
# === BEFORE (original) ===
# out = CustomModel(**inputs,labels = target)

# === AFTER (edited) ===
# labels for multiple-choice must be class indices of shape (batch_size,)
# use the existing `labels` tensor created above (value 0)
out = CustomModel(**inputs, labels=labels)
out

ValueError: Expected input batch_size (2) to match target batch_size (1).

In [14]:
import torch
import torch.nn as nn

out = CustomModel(**inputs, labels=target)

# Multiple-choice contract: one score per choice
assert out.logits.ndim == 2
assert out.logits.shape[1] == CustomModel.num_choice

# Label contract: one correct choice per sample (single-label)
target_idx = torch.argmax(target, dim=1)
assert target_idx.shape == (out.logits.shape[0],)

# Expected loss behavior for this benchmark: NLLLoss on class indices
expected = nn.NLLLoss()(out.logits.view(-1, CustomModel.num_choice), target_idx.view(-1))

assert torch.isfinite(out.loss).item()
assert torch.allclose(out.loss, expected, atol=1e-6), \
    "Loss must match single-label class-index objective (argmax-converted labels)."

ValueError: Expected input batch_size (2) to match target batch_size (3).